# ML-эксперименты E1–E6 — дипломная работа

Пошаговый прогон шести экспериментов поверх обученной v3-модели (val_acc=0.726):

| # | Эксперимент | Требует весов | Требует GPU | Время |
|---|---|---|---|---|
| E3 | Score-card per-class | нет | нет | ~5 мин |
| E1 | Grad-CAM | да | желательно | ~5 мин CPU / ~1 мин GPU |
| E2 | Embeddings + t-SNE/UMAP | да | желательно | ~5 мин |
| E4 | Калибровка вероятностей | да | нет | ~3 мин |
| E5 | CLIP linear-probe + zero-shot | нет (CLIP качается) | желательно | ~10 мин GPU |
| E6 | Multi-label fine-tune | warm-start от v3 | **обязательно** | ~30 мин T4 |

**Включить GPU:** Runtime → Change runtime type → **T4 GPU**.

## Что должно лежать в Drive

```
MyDrive/
├── dataset/                              ← плоская структура по классам
│   ├── airy/, dark/, dramatic/,
│   ├── golden_hour/, minimalist/,
│   ├── monochrome/, neon/, vintage/
└── training_out_v3/
    └── efficientnet_b0_styles.pth        ← обученные веса
```

Артефакты будут в `MyDrive/diploma_out/`.

## Воспроизведение оригинального split'а

В блокноте `train_pipeline.ipynb` v3 обучалась на split'е с такой методологией:
- MD5-дедупликация внутри каждого класса,
- cap каждого класса до 600 (`minimalist` — отдельно до 500, потому что был раздут до 1457),
- random_split 80/20 с `seed=42`.

Скрипт `training/make_split.py` поддерживает эти опции и воспроизводит
тот же split (используется в ячейке 6 ниже).

## Шаг 0. Setup — один раз в начале сессии

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Клонируем ветку с экспериментами
!rm -rf /content/visual-style-classifier
!git clone -b claude/sleepy-gould-93f13f \
    https://github.com/tvrvs91/visual-style-classifier.git /content/visual-style-classifier
%cd /content/visual-style-classifier

In [ ]:
# 3) Доп. зависимости (поверх того что в Colab уже стоит)
!pip install -q -r training/requirements-experiments.txt

In [ ]:
# 4) Переменные путей
import os
DRIVE = '/content/drive/MyDrive'
WEIGHTS = f'{DRIVE}/training_out_v3/efficientnet_b0_styles.pth'
DATA = '/content/dataset'                 # сюда положим split (симлинки)
OUT = f'{DRIVE}/diploma_out'              # артефакты экспериментов на Drive
os.makedirs(OUT, exist_ok=True)

assert os.path.exists(WEIGHTS), f'нет файла весов: {WEIGHTS}'
assert os.path.exists(f'{DRIVE}/dataset'), f'нет датасета: {DRIVE}/dataset'
print('Setup OK')
print(f'  weights : {WEIGHTS}')
print(f'  dataset : {DRIVE}/dataset')
print(f'  out     : {OUT}')

In [ ]:
# 5) Воспроизводим v3 split (MD5-dedup + cap 600 + minimalist 500 + 80/20 seed=42)
#    Drive не меняется — split складывается в /content/dataset/ через симлинки.
!python training/make_split.py \
    --src "$DRIVE/dataset" \
    --dst {DATA} \
    --val-ratio 0.20 --seed 42 \
    --dedup \
    --target-count 600 \
    --cap-class minimalist:500

## Шаг 0bis. Sanity-check: подтвердить acc v3 на нашем split'е

Прогоняем `diagnose.py` на тех же весах и нашем split'е. Ожидается
acc ≈ 0.72–0.73 (оригинал был 0.726). Если сильно меньше — что-то с
путями/датасетом.

In [ ]:
!python training/diagnose.py \
    --weights {WEIGHTS} \
    --val-dir {DATA}/val \
    --arch efficientnet_b0 \
    --out-dir {OUT}/diagnose_v3

## E3. Score-card per-class анализ

Самый быстрый, GPU не нужен. Прогоняем первым.

**Что получаем:**
- `radar.png` — 8 радарных чартов
- `violins.png` — распределения 5 метрик по классам
- `stats.csv` — mean/std/median
- `anova.json` — значимость метрик
- `baseline.json` — accuracy logreg/RF на 5 фичах

In [ ]:
!python training/score_card_class.py \
    --data-dir {DATA} \
    --out-dir {OUT}/scorecard \
    --splits train val

In [ ]:
from IPython.display import Image, display
import json
display(Image(f'{OUT}/scorecard/radar.png'))
display(Image(f'{OUT}/scorecard/violins.png'))
print(json.dumps(json.load(open(f'{OUT}/scorecard/baseline.json')), indent=2, ensure_ascii=False)[:1500])

## E1. Grad-CAM визуализация attention

**Что получаем:**
- `gradcam_grid.png` — общий 8×6 grid
- `gradcam_<class>.png` — отдельные коллажи по классам
- `gradcam_errors.png` — топ-10 уверенно неправильных
- `notes.json` — метаданные

In [ ]:
!python training/gradcam_viz.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/gradcam \
    --per-class 6 --errors 10

In [ ]:
display(Image(f'{OUT}/gradcam/gradcam_grid.png'))
display(Image(f'{OUT}/gradcam/gradcam_errors.png'))

## E2. Embeddings + t-SNE / UMAP

**Что получаем:**
- `tsne.png`, `umap.png` — 2D-проекции
- `metrics.json` — silhouette, Davies-Bouldin, kNN5-accuracy
- `centroid_similarity.png/.csv` — матрица 8×8
- `embeddings.npy`, `labels.npy` — сырые векторы

In [ ]:
!python training/embed_viz.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/embed \
    --split val --umap

In [ ]:
display(Image(f'{OUT}/embed/tsne.png'))
display(Image(f'{OUT}/embed/umap.png'))
display(Image(f'{OUT}/embed/centroid_similarity.png'))
print(json.dumps(json.load(open(f'{OUT}/embed/metrics.json')), indent=2, ensure_ascii=False))

## E4. Калибровка вероятностей (temperature scaling)

Метод Guo et al. 2017. Подбирает T так, чтобы softmax(logits/T) был лучше калиброван.

**Что получаем:**
- `reliability_combined.png` — до/после
- `metrics.json` — ECE/MCE/Brier/NLL
- `temperature.txt` — найденное T

In [ ]:
!python training/calibration.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/calibration \
    --vector-scaling

In [ ]:
display(Image(f'{OUT}/calibration/reliability_combined.png'))
print('Найденное T:', open(f'{OUT}/calibration/temperature.txt').read())
print(json.dumps(json.load(open(f'{OUT}/calibration/metrics.json')), indent=2, ensure_ascii=False))

## E5. CLIP linear-probe + zero-shot

Сравнение с CLIP ViT-B/32 (OpenAI). Качает ~350 МБ модели. Включи GPU.

**Что получаем:**
- `comparison_table.csv`
- `confusion_matrix_lp.png`, `confusion_matrix_zs.png`
- `tsne_clip.png`

In [ ]:
!python training/clip_probe.py \
    --data-dir {DATA} \
    --out-dir {OUT}/clip \
    --model ViT-B-32 --pretrained openai

In [ ]:
import csv
with open(f'{OUT}/clip/comparison_table.csv') as f:
    for row in csv.reader(f):
        print(row)
display(Image(f'{OUT}/clip/confusion_matrix_lp.png'))
display(Image(f'{OUT}/clip/tsne_clip.png'))

## E6. Multi-label fine-tune (опционально, требует GPU)

Тёплый старт от v3-весов. ~30 мин на T4.

**Что получаем:**
- `efficientnet_b0_multilabel.pth`
- `thresholds.json` — per-class threshold
- `metrics.json` — mAP, per-class F1, exact-match, Hamming
- `co_label_stats.json` — комбинации меток

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
!python training/train_multilabel.py \
    --data-dir {DATA} \
    --out-dir {OUT}/multilabel \
    --init-weights {WEIGHTS} \
    --epochs 25 --head-epochs 5 \
    --batch-size 32 --num-workers 2 \
    --patience 7

In [ ]:
metrics = json.load(open(f'{OUT}/multilabel/metrics.json'))
print('mAP:', metrics['mAP'])
print('macro F1:', metrics['macro_f1'])
print('exact_match:', metrics['exact_match_accuracy'])
print('hamming:', metrics['hamming_accuracy'])
print()
print('Per-class:')
for cls, m in metrics['per_class'].items():
    print(f"  {cls:14s}  F1={m['f1']:.3f}  AP={m['AP']:.3f}  thr={m['threshold']:.2f}")

## Финальная сводка

После прохода всех ячеек в `MyDrive/diploma_out/` будет:

```
diploma_out/
├── diagnose_v3/  (sanity-check acc на нашем split'е)
├── scorecard/    (E3)
├── gradcam/      (E1)
├── embed/        (E2)
├── calibration/  (E4)
├── clip/         (E5)
└── multilabel/   (E6, опционально)
```

Дальше пришли путь — я прочитаю результаты и переведу их в текст для дипломного отчёта.